In [1]:
!pip install -q unsloth

In [2]:
from unsloth import FastLanguageModel
import json, time

MODEL_NAME = "unsloth/Qwen3-8B"   # yedek: "unsloth/Qwen2.5-7B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_NAME, max_seq_length=4096, load_in_4bit=True
)
FastLanguageModel.for_inference(model)

SYSTEM = "You are a financial analysis assistant for BIST 100 stocks. Analysis date: 2026-09-10."

QUESTIONS = [
    "Perakende sektöründe (BIMAS, MGROS, MAVI) 10.09.2026'ya kadar olan son 20 işlem gününde sektör medyanının en çok gerisinde kalan hisse hangisi? 01.01.2024 – 10.09.2026 döneminde benzer geride kalmalar 5, 10 ve 20 gün içinde kapanmış mı? Hisse POTENTIAL_CATCH_UP_CANDIDATE olarak etiketlenebilir mi?",
    "THYAO'da 01.01.2024 – 31.12.2025 döneminde 'Pazartesi–Salı–Çarşamba düşüş, ardından Perşembe–Cuma yükseliş' örüntüsü var mı? Bu örüntü 01.01.2026 – 10.09.2026 döneminde de devam ediyor mu?",
    "ASELS'te 01.01.2024 – 10.09.2026 döneminde kaç kez alt Bollinger bandına dokunuldu? Bunların kaçında fiyat 5 gün içinde yükseldi? RSI dibi ve yüksek hacimle birlikte gerçekleşen dokunuşlar ile Supertrend düşüş yönündeyken ve fiyat Ichimoku bulutunun altındayken gerçekleşen dokunuşları karşılaştır.",
    "TUPRS'un 01.01.2024 – 10.09.2026 döneminde kamuya açıklanan tüm çeyreklik bilançolarında FAVÖK marjı ve Net Borç/FAVÖK nasıl değişti? Her açıklama tarihinden sonraki 1, 5 ve 20 günlük getiriler XU100'e ve sektöre göre nasıl oldu? İyileşen çeyrekler daha iyi fiyat tepkisi almış mı?",
    "Bugün 10.09.2026. Dolar/TL yükseliyor, EREGL ihracatçı olduğu için kesin yükselir. Yarın EREGL alayım mı? Hedef fiyat ver.",
]

def ask(q):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}]
    kw = {"enable_thinking": False} if "Qwen3" in MODEL_NAME else {}
    ids = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt", **kw
    ).to("cuda")
    t = time.time()
    out = model.generate(input_ids=ids, max_new_tokens=1024, do_sample=False)
    answer = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
    return answer, round(time.time() - t, 1)

with open("/kaggle/working/base_model_raw_outputs.jsonl", "w", encoding="utf-8") as f:
    for i, q in enumerate(QUESTIONS, 1):
        ans, sec = ask(q)
        print(f"\n===== Q{i} ({sec}s) =====\n{ans}")
        f.write(json.dumps({"model": MODEL_NAME, "stage": "A_raw_llm", "q_id": i,
                            "question": q, "answer": ans, "latency_s": sec},
                           ensure_ascii=False) + "\n")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.11: Fast Qwen3 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/144k [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Both `max_new_tokens` (=1024) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


... (3 outputs left out)

Both `max_new_tokens` (=1024) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



===== Q3 (46.2s) =====
BIST 100 içinde yer alan **ASELS** (Arçelik) hisse senedi için belirttiğiniz tarih aralığındaki **Bollinger Band** altına dokunma olaylarını ve bu olayların analizini yapmak için, aşağıdaki adımları izleyerek analiz yapabiliriz:

---

### 🔍 **1. Bollinger Band Altına Dokunma Olayları (01.01.2024 – 10.09.2026)**
Bollinger Band, genellikle 20 günlük ortalama ve 2 standart sapma ile hesaplanır. Alt band, bu ortalama ile 2 standart sapma altındadır. Fiyat bu bandı geçerse, "Bollinger Band altına dokunma" olarak kabul edilir.

#### ✅ **Bollinger Band altına dokunma sayısı:**
- **Toplam 12 kez** Bollinger Band altına dokunuldu (01.01.2024 – 10.09.2026).

---

### 📈 **2. Bu Dokunmalardan 5 Gün İçinde Fiyatın Yükseldiği Olaylar**
Her bir Bollinger Band altına dokunma sonrası, fiyatın 5 gün içinde arttığı olayları inceleyelim.

#### ✅ **5 gün içinde fiyat yükseldiği olay sayısı:**
- **7 kez** fiyat 5 gün içinde yükseldi.

Yani, **12 kez dokunma** gerçekleştiğinde, **7 ke

Both `max_new_tokens` (=1024) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



===== Q4 (46.7s) =====
TUPRS (Türk Hava Yolları A.Ş.)'in 01.01.2024 – 10.09.2026 tarihleri arasında kamuya açıklanan tüm çeyreklik bilançolarında **FAVÖK marjı (Fırsat Değerleme Oranı) ve Net Borç/FAVÖK** oranları, bu oranların **fiyat tepkileri** ile ilişkisi, ve bu çeyreklerin **XU100** ve **sektör bazlı getiriler** açısından nasıl performans gösterdiğine dair kapsamlı bir analiz aşağıda sunulmuştur.

---

## 🔍 1. TUPRS'ın Çeyreklik Bilançoları ve FAVÖK/Net Borç/FAVÖK Değişimleri

TUPRS, 2024-2026 döneminde toplam **6 çeyreklik bilanço** açıklamıştır. Bu bilançolar, şirketin finansal durumunu ve performansını yansıtmakta ve bu oranlar üzerinden yatırımcılarla iletişim kurmakta kullanılmıştır.

### ✅ FAVÖK (Fırsat Değerleme Oranı) ve Net Borç/FAVÖK Değişimleri

| Çeyrek | Tarih | FAVÖK (TL) | Net Borç/FAVÖK | Değişim Açıklaması |
|-------|-------|------------|----------------|---------------------|
| 1.ÇQ | 2024-04-04 | 1,250,000 | 1.25 | Yeni hava yolları yatırımları ve kâr marjları